In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install timm

In [3]:
import torch
import torch.nn as nn
import timm
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
class_mapping = {
    "Aphid": 0,
    "Black Rust": 1,
    "Blast": 2,
    "Fusarium Head Blight": 3,
    "Brown Rust": 4,
    "Common Root Rot": 5,
    "Healthy": 6,
    "Leaf Blight": 7,
    "Mildew": 8,
    "Mite": 9,
    "Septoria": 10,
    "Smut": 11,
    "Stem fly": 12,
    "Tan spot": 13,
    "Yellow Rust": 14,
}

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [23]:
# class PlantDocDataset(Dataset):
#     def __init__(self, root, transform=None):
#         self.samples = []
#         self.transform = transform

#         for folder_name in os.listdir(root):
#             mapped_label = None
            
#             if folder_name in class_mapping:
#                 mapped_label = class_mapping[folder_name]
            
#             else:
#                 clean_folder = folder_name.lower().replace('_valid', '').replace('_', ' ')
                
#                 for key, val in class_mapping.items():
#                     clean_key = key.lower().replace('_', ' ')
#                     if clean_key == clean_folder:
#                         mapped_label = val
#                         break

#             if mapped_label is not None:
#                 class_path = os.path.join(root, folder_name)
#                 for img in os.listdir(class_path):
#                     if img.lower().endswith(('.png', '.jpg', '.jpeg')):
#                         self.samples.append(
#                             (os.path.join(class_path, img), mapped_label)
#                         )

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         img_path, label = self.samples[idx]
#         image = Image.open(img_path).convert("RGB")

#         if self.transform:
#             image = self.transform(image)

#         return image, label

In [12]:
train_root = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"
val_root = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/valid"

pd_train_dataset = PlantDocDataset(
    train_root,
    transform=train_transform
)

pd_val_dataset = PlantDocDataset(
    val_root,
    transform=val_transform
)

print("Dataset train size:", len(pd_train_dataset))
print("Dataset val size:", len(pd_val_dataset))

Dataset train size: 13104
Dataset val size: 280


In [14]:
train_loader = DataLoader(
    pd_train_dataset,  
    batch_size=16,
    shuffle=True,     
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    pd_val_dataset, 
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

num_classes = 15

In [16]:
model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=True,
    img_size=224
)

for param in model.parameters():
    param.requires_grad = False

for param in model.blocks[-2:].parameters():
    param.requires_grad = True

in_features = model.num_features

model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [17]:
from collections import Counter
import torch
import torch.nn as nn

train_labels = [label for _, label in pd_train_dataset.samples] 

class_counts = Counter(train_labels)

weights = [1.0 / max(class_counts.get(i, 1), 1) for i in range(num_classes)]
weights = torch.tensor(weights).float().to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [18]:
from torch.amp import GradScaler, autocast

def train_model(model, train_loader, val_loader, epochs=30, patience=10):

    scaler = GradScaler("cuda")
    best_acc = 0
    early_stop = 0

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                with autocast("cuda"):
                    outputs = model(images)

                _, preds = torch.max(outputs,1)

                total += labels.size(0)
                correct += (preds==labels).sum().item()

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_wheat_finetuned.pth")
            early_stop = 0
        else:
            early_stop += 1

        if early_stop >= patience:
            print("Early stopping triggered.")
            break

        scheduler.step()

    print("Best Validation Accuracy:", best_acc)

In [19]:
train_model(model, train_loader, val_loader, epochs=30)

Epoch 1: Loss=1.0138 | Val Acc=0.7964
Epoch 2: Loss=0.6121 | Val Acc=0.8393
Epoch 3: Loss=0.5034 | Val Acc=0.8679
Epoch 4: Loss=0.4482 | Val Acc=0.8821
Epoch 5: Loss=0.3974 | Val Acc=0.8929
Epoch 6: Loss=0.3635 | Val Acc=0.8857
Epoch 7: Loss=0.3318 | Val Acc=0.8821
Epoch 8: Loss=0.3184 | Val Acc=0.9179
Epoch 9: Loss=0.3016 | Val Acc=0.9143
Epoch 10: Loss=0.2851 | Val Acc=0.9036
Epoch 11: Loss=0.2821 | Val Acc=0.9143
Epoch 12: Loss=0.2686 | Val Acc=0.9000
Epoch 13: Loss=0.2670 | Val Acc=0.9071
Epoch 14: Loss=0.2640 | Val Acc=0.9000
Epoch 15: Loss=0.2561 | Val Acc=0.9250
Epoch 16: Loss=0.2519 | Val Acc=0.9214
Epoch 17: Loss=0.2464 | Val Acc=0.9214
Epoch 18: Loss=0.2449 | Val Acc=0.9214
Epoch 19: Loss=0.2383 | Val Acc=0.9250
Epoch 20: Loss=0.2391 | Val Acc=0.9179
Epoch 21: Loss=0.2336 | Val Acc=0.9214
Epoch 22: Loss=0.2331 | Val Acc=0.9286
Epoch 23: Loss=0.2282 | Val Acc=0.9179
Epoch 24: Loss=0.2264 | Val Acc=0.9250
Epoch 25: Loss=0.2257 | Val Acc=0.9214
Epoch 26: Loss=0.2244 | Val Acc=0.

In [24]:
class PlantDocDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for folder_name in os.listdir(root):
            mapped_label = None
            
            if folder_name in class_mapping:
                mapped_label = class_mapping[folder_name]
            
            else:
                clean_folder = folder_name.lower().replace('_valid', '').replace('_test', '').replace('_', ' ')
                
                for key, val in class_mapping.items():
                    clean_key = key.lower().replace('_', ' ')
                    if clean_key == clean_folder:
                        mapped_label = val
                        break

            if mapped_label is not None:
                class_path = os.path.join(root, folder_name)
                for img in os.listdir(class_path):
                    if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append(
                            (os.path.join(class_path, img), mapped_label)
                        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [25]:
test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

plantdoc_test_root = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/test"

pd_test_dataset = PlantDocDataset(
    plantdoc_test_root,
    transform=test_transform
)

test_loader = DataLoader(
    pd_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Rice dataset Test size:", len(pd_test_dataset))

Rice dataset Test size: 750


In [26]:
import timm
import torch.nn as nn

num_classes = 15

model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    img_size=224
)

in_features = model.num_features
model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)
model.load_state_dict(
    torch.load("best_wheat_finetuned.pth", map_location=device)
)

model = model.to(device)
model.eval()

print("Best DINO model loaded successfully.")

Best DINO model loaded successfully.


In [28]:
from sklearn.metrics import confusion_matrix, classification_report

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print("\n==============================")
print("DINOv2 Test Accuracy:", test_acc)
print("==============================")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

class_names = [
    "Aphid",
    "Black Rust",
    "Blast",
    "Fusarium Head Blight",
    "Brown Rust",
    "Common Root Rot",
    "Healthy",
    "Leaf Blight",
    "Mildew",
    "Mite",
    "Septoria",
    "Smut",
    "Stem fly",
    "Tan spot",
    "Yellow Rust",
]

print("\nPer Class Accuracy:")

for i, class_name in enumerate(class_names):
    class_total = cm[i].sum()
    class_correct = cm[i][i]

    acc = class_correct / class_total if class_total > 0 else 0
    print(f"{class_name}: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))


DINOv2 Test Accuracy: 0.9173333333333333

Confusion Matrix:
[[50  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 45  0  0  5  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 50  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 50  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 50  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 50  0  0  0  0  0  0  0  0  0]
 [ 0  0  2  0  0  0  5  0  0  0  0  0  0  0 43]
 [ 0  1  0  0  0  0  0 45  0  0  0  0  0  4  0]
 [ 0  0  0  0  0  0  0  0 49  0  0  0  1  0  0]
 [ 1  0  0  0  0  0  0  0  0 48  0  0  0  1  0]
 [ 0  0  0  0  0  0  0  4  0  0 46  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 50  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 50  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0 50  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0 50]]

Per Class Accuracy:
Aphid: 1.0000
Black Rust: 0.9000
Blast: 1.0000
Fusarium Head Blight: 1.0000
Brown Rust: 1.0000
Common Root Rot: 1.0000
Healthy: 0.1000
Leaf Blight: 0.9000
Mildew: 0.9800
Mite: 0.9600
Septoria: 0.92